In [2]:
import json
import time
from itertools import product

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [12]:
# data source 1: ETER university funding and outcomes
eu_country_codes = [
    'AT','BE','BG','HR','CY','CZ','DK','EE','FI','FR',
    'DE','GR','HU','IE','IT','LV','LT','LU','MT','NL',
    'PL','PT','RO','SK','SI','ES','SE'
]

eter_raw = pd.read_csv('../datafiles/university_funding.csv')

df = eter_raw[
    eter_raw['Country_Code'].isin(eu_country_codes) &
    (eter_raw['Reference_year'] == 2020)
].reset_index(drop=True)
print(f'ETER EU 2020 shape: {df.shape}')

funding_cols = [
    'Total_Current_revenues_(EURO)',
    'Basic_government_allocation_(EURO)',
    'Student_fees_funding_(EURO)',
    'Total_third_party_funding_(EURO)',
    'Personnel_expenditure_(EURO)',
    'R&D_Expenditure_(EURO)'
]
result_cols = [
    'Total_students_enrolled_ISCED_5_7',
    'Total_graduates_ISCED_5_7',
    'Total_graduates_at_ISCED_8'
]
keeping_cols = [
    'ETER_ID', 'English_Institution_Name', 'Institution_Name',
    'Reference_year', 'Country_Code', 'Name_of_the_city'
] + funding_cols + result_cols

df = df[keeping_cols].copy()

for col in funding_cols + result_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.reset_index(drop=True, inplace=True)
print(df.shape)
print(df[funding_cols + result_cols].describe().to_string())

ETER EU 2020 shape: (2033, 39)
(2033, 15)
       Total_Current_revenues_(EURO)  Basic_government_allocation_(EURO)  Student_fees_funding_(EURO)  Total_third_party_funding_(EURO)  Personnel_expenditure_(EURO)  R&D_Expenditure_(EURO)  Total_students_enrolled_ISCED_5_7  Total_graduates_ISCED_5_7  Total_graduates_at_ISCED_8
count                   8.270000e+02                        3.510000e+02                 7.350000e+02                      7.920000e+02                  7.880000e+02            3.170000e+02                        1742.000000                1746.000000                  847.000000
mean                    1.088374e+08                        9.765850e+07                 7.682322e+06                      2.231926e+07                  7.488863e+07            1.460837e+07                        8310.347325                1753.530928                  100.517119
std                     1.723579e+08                        1.233880e+08                 2.085014e+07                 

In [13]:
# data source 2: QS world university rankings
eu_country_names = [
    'Austria','Belgium','Bulgaria','Croatia','Cyprus','Czech Republic',
    'Denmark','Estonia','Finland','France','Germany','Greece','Hungary',
    'Ireland','Italy','Latvia','Lithuania','Luxembourg','Malta','Netherlands',
    'Poland','Portugal','Romania','Slovakia','Slovenia','Spain','Sweden'
]

qs_raw = pd.read_excel('../datafiles/university_world_rankings.xlsx', header=1)
qs_raw.columns = qs_raw.iloc[0]
qs_raw = qs_raw.drop(0).reset_index(drop=True)

df_2 = qs_raw[qs_raw['Country/Territory'].isin(eu_country_names)].reset_index(drop=True)
df_2.to_csv('../datafiles/university_world_rankings.csv', index=False)
print(f'QS EU shape: {df_2.shape}')

qs_score_cols = ['Overall SCORE', 'AR SCORE', 'ER SCORE', 'FSR SCORE',
                 'CPF SCORE', 'IFR SCORE', 'ISR SCORE', 'EO SCORE', 'SUS SCORE']
for col in qs_score_cols:
    df_2[col] = pd.to_numeric(df_2[col], errors='coerce')

print(df_2[['Name', 'Country/Territory'] + qs_score_cols].head(10).to_string())

QS EU shape: (311, 31)
0                                               Name Country/Territory  Overall SCORE  AR SCORE  ER SCORE  FSR SCORE  CPF SCORE  IFR SCORE  ISR SCORE  EO SCORE  SUS SCORE
0                     Technical University of Munich           Germany           90.2      92.0      99.7       70.5       92.6       86.3       98.9      57.3       87.8
1                                     PSL University            France           88.6      82.6      98.5       98.6       85.1       69.6       75.5      97.5       87.8
2                    Institut Polytechnique de Paris            France           85.4      58.1      99.8       94.8       97.3       99.9       99.2      99.1       77.9
3                     Delft University of Technology       Netherlands           84.3      86.2      90.4       49.1       84.6      100.0       94.8      66.0       96.3
4                        The University of Amsterdam       Netherlands           81.5      92.5      72.6       27.5      

In [16]:
qs_keep = ['Name'] + qs_score_cols + ['Rank', 'Size', 'Focus', 'Research', 'Status']
qs_merge = df_2[[c for c in qs_keep if c in df_2.columns]].copy()

merge1 = pd.merge(df, qs_merge, left_on='English_Institution_Name', right_on='Name', how='inner')
print(f'Merge (English name): {len(merge1)} rows')

unmatched = df[~df['English_Institution_Name'].isin(merge1['English_Institution_Name'])]
merge2 = pd.merge(unmatched, qs_merge, left_on='Institution_Name', right_on='Name', how='inner')
print(f'Merge (local name): {len(merge2)} rows')

merged = pd.concat([merge1, merge2], ignore_index=True)

merged['graduation_rate'] = (
    merged['Total_graduates_ISCED_5_7'] / merged['Total_students_enrolled_ISCED_5_7']
).clip(0, 1)

print(f'\nFinal merged shape: {merged.shape}')
print(merged[['English_Institution_Name', 'Country_Code',
              'Total_Current_revenues_(EURO)', 'Overall SCORE',
              'graduation_rate']].head(10).to_string())

merged.to_csv('../datafiles/budget_funding_plan.csv', index=False)
print('\nSaved to budget_funding_plan.csv')

Merge (English name): 115 rows
Merge (local name): 62 rows

Final merged shape: (177, 31)
          English_Institution_Name Country_Code  Total_Current_revenues_(EURO)  Overall SCORE  graduation_rate
0             University of Vienna           AT                   6.336029e+08           61.9         0.130345
1  Vienna University of Technology           AT                   3.929993e+08           55.8         0.131609
2    Graz University of Technology           AT                   2.644602e+08           36.0         0.147610
3         University of Klagenfurt           AT                   8.026591e+07           25.1         0.145464
4              University of Namur           BE                            NaN            NaN         0.210627
5               University of Mons           BE                            NaN            NaN         0.233281
6                        KU Leuven           BE                   1.150651e+09           79.7         0.315724
7               Hassel